In [6]:
"""
Enhanced Voice Cloning Pipeline for British Audiobook Voice
Optimized for Mac M3 Pro with CPU training and fine-tuning
"""

import os
import json
import librosa
import soundfile as sf
import pandas as pd
from pathlib import Path
import numpy as np
from typing import List, Tuple, Optional
import logging
from concurrent.futures import ThreadPoolExecutor
import multiprocessing as mp
from tqdm import tqdm
import time

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class AudioDataProcessor:
    """Process and validate audio data for TTS fine-tuning"""
    
    def __init__(self, audio_dir: str, csv_path: str, output_dir: str):
        self.audio_dir = Path(audio_dir)
        self.csv_path = Path(csv_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        # Audio quality thresholds
        self.min_duration = 1.0  # seconds
        self.max_duration = 30.0  # seconds
        self.min_sample_rate = 16000
        self.target_sample_rate = 22050  # Common for TTS
        
    def load_transcriptions(self) -> pd.DataFrame:
        """Load transcriptions from CSV"""
        try:
            df = pd.read_csv(self.csv_path, delimiter="|")
            logger.info(f"Loaded {len(df)} transcriptions from CSV")
            return df
        except Exception as e:
            logger.error(f"Error loading CSV: {e}")
            raise
    
    def validate_audio_file(self, audio_path: Path) -> Optional[dict]:
        """Validate single audio file and return metadata"""
        try:
            # Load audio
            audio, sr = librosa.load(audio_path, sr=None)
            duration = len(audio) / sr
            
            # Basic quality checks
            if duration < self.min_duration or duration > self.max_duration:
                return None
                
            if sr < self.min_sample_rate:
                return None
                
            # Check for silence (RMS energy)
            rms = librosa.feature.rms(y=audio)[0]
            avg_rms = np.mean(rms)
            if avg_rms < 0.001:  # Very quiet audio
                return None
                
            # Check for clipping
            if np.max(np.abs(audio)) > 0.99:
                logger.warning(f"Possible clipping in {audio_path}")
                
            return {
                'path': str(audio_path),
                'duration': duration,
                'sample_rate': sr,
                'rms': avg_rms,
                'max_amplitude': np.max(np.abs(audio))
            }
            
        except Exception as e:
            logger.error(f"Error processing {audio_path}: {e}")
            return None
    
    def clean_text(self, text: str) -> str:
        """Clean transcription text for TTS training"""
        if not text:
            return ""
            
        import re
        
        # Remove quotes at beginning and end
        text = str(text)
        text = text.strip('"\'')
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text)
        
        # Remove or normalize special characters while preserving punctuation
        text = re.sub(r'[^\w\s\.\,\!\?\;\:\-\'\"]', '', text)
        
        # Ensure proper sentence endings
        if text and text[-1] not in '.!?':
            text += '.'
        
        return text.strip()
    
    def process_dataset(self) -> pd.DataFrame:
        """Process entire dataset and create training manifest"""
        # Load transcriptions
        transcriptions_df = self.load_transcriptions()
        
        logger.info("Processing audio files...")
        valid_samples = []
        
        # Process each transcription entry
        for idx, row in tqdm(transcriptions_df.iterrows(), total=len(transcriptions_df), desc="Processing files"):
            audio_path = self.audio_dir / row['file_name']
            
            # Skip if audio file doesn't exist
            if not audio_path.exists():
                logger.warning(f"Audio file not found: {audio_path}")
                continue
            
            # Validate audio
            audio_info = self.validate_audio_file(audio_path)
            if not audio_info:
                continue
            
            # Clean transcription
            clean_text = self.clean_text(row['transcription'])
            if len(clean_text) < 10:  # Too short
                continue
            
            # Add to valid samples
            valid_samples.append({
                'audio_path': str(audio_path),
                'text': clean_text,
                'duration': audio_info['duration'],
                'sample_rate': audio_info['sample_rate'],
                'speaker_id': 'benedict_cumberbatch'
            })
        
        logger.info(f"Valid samples: {len(valid_samples)} out of {len(transcriptions_df)}")
        
        # Create DataFrame
        df = pd.DataFrame(valid_samples)
        
        # Save manifest
        manifest_path = self.output_dir / 'training_manifest.json'
        df.to_json(manifest_path, orient='records', indent=2)
        
        # Save CSV for easy viewing
        csv_path = self.output_dir / 'training_manifest.csv'
        df.to_csv(csv_path, index=False)
        
        logger.info(f"Manifest saved to {manifest_path}")
        return df
    
    def prepare_for_coqui_tts(self, df: pd.DataFrame):
        """Prepare data in Coqui TTS format with fine-tuning considerations"""
        # Create Coqui TTS metadata format
        metadata = []
        
        for _, row in df.iterrows():
            # Coqui format: audio_path|text|speaker_name
            metadata.append(f"{row['audio_path']}|{row['text']}|benedict_cumberbatch")
        
        # Save metadata file
        metadata_path = self.output_dir / 'metadata.txt'
        with open(metadata_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(metadata))
        
        logger.info(f"Coqui TTS metadata saved to {metadata_path}")
        
        # Create config for fine-tuning (conservative settings to preserve base model)
        config = {
            "model_name": "tts_models/en/ljspeech/tacotron2-DDC",
            "dataset": "custom",
            "audio_path": str(self.audio_dir),
            "metadata_path": str(metadata_path),
            "output_path": str(self.output_dir / 'training_output'),
            "sample_rate": 22050,
            "batch_size": 16,  # Conservative for CPU and fine-tuning
            "epochs": 100,  # Fewer epochs for fine-tuning
            "learning_rate": 0.0001,  # Lower learning rate for fine-tuning
            "eval_split_size": 0.1,
            "print_step": 50,  # Less frequent printing
            "save_step": 500,  # Regular checkpoints
            "eval_step": 250,  # Evaluation frequency
            "sample_step": 500,  # Sample generation frequency (half epoch approximately)
            "warmup_steps": 200,
            "weight_decay": 1e-6,
            "grad_clip": 5.0,
            "mixed_precision": False,  # Keep False for CPU
            "use_phonemes": False,
            "fine_tuning": True,  # Enable fine-tuning mode
            "freeze_encoder": False,  # Can freeze encoder for more conservative fine-tuning
            "freeze_decoder": False
        }
        
        config_path = self.output_dir / 'config.json'
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)
        
        logger.info(f"Fine-tuning config saved to {config_path}")


class CoquiTTSFineTuner:
    """Handle Coqui TTS fine-tuning with progress tracking and sampling"""
    
    def __init__(self, config_path: str):
        self.config_path = config_path
    
    def install_dependencies(self):
        """Install required packages"""
        install_commands = [
            "pip install coqui-tts>=0.20.0",
            "pip install librosa soundfile pandas tqdm",
            "pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu",
            "pip install tensorboard"  # For monitoring
        ]
        
        print("Run these commands to install dependencies:")
        for cmd in install_commands:
            print(f"  {cmd}")
    
    def create_training_script(self, output_path: str):
        """Create enhanced fine-tuning script with progress tracking"""
        
        training_script = '''#!/usr/bin/env python3
"""
Enhanced Coqui TTS Fine-tuning Script with Progress Tracking
Designed for conservative fine-tuning to preserve base model quality
"""

import os
import json
import time
from pathlib import Path
import torch
from tqdm import tqdm
import re
import logging

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="TTS.tts.models.tacotron2")
warnings.filterwarnings("ignore", category=UserWarning, module="torch.functional")
warnings.filterwarnings("ignore", category=UserWarning, module="pkg_resources")
warnings.filterwarnings("ignore", category=UserWarning, module="jieba")

from TTS.utils.manage import ModelManager
from TTS.tts.configs.tacotron2_config import Tacotron2Config
from TTS.tts.models.tacotron2 import Tacotron2
from trainer import Trainer, TrainerArgs
from TTS.tts.datasets import load_tts_samples
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.synthesis import synthesis
from TTS.config import BaseDatasetConfig


# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def custom_formatter(root_path, manifest_path, **kwargs):
    """Custom formatter with GST support"""
    metadata_path = Path(manifest_path)
    items = []
    with open(metadata_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith("file_name"):  # Skip header
                continue
            parts = line.strip().split('|')
            file_name = parts[0].strip('"')
            text = '|'.join(parts[1:]).strip('"')
            wav_path = str(Path(root_path) / f"{file_name}.wav")
            items.append({
                "text": text,
                "audio_file": wav_path,
                "speaker": "benedict",
                "style_wav": wav_path  # Use same audio for GST reference
            })
    return items

class ProgressTracker:
    """Track training progress and generate samples"""
    
    def __init__(self, model, config, ap, tokenizer, output_path, style_reference_path):
        self.model = model
        self.config = config
        self.ap = ap  
        self.tokenizer = tokenizer
        self.output_path = Path(output_path)
        self.samples_dir = self.output_path / 'samples'
        self.samples_dir.mkdir(exist_ok=True)
        self.style_reference_path = style_reference_path
        
        # Sample texts for generation
        self.sample_texts = [
            "Hello, this is a test of the fine-tuned Benedict Cumberbatch voice model.",
            "The quality of mercy is not strained, it droppeth as the gentle rain from heaven.",
            "My dear Watson, the game is afoot and adventure awaits us both.",
            "In 1715 my father fell madly in love with an actress named Fragoletha.",
            "Elementary, my dear fellow, the solution is quite obvious when you observe carefully."
        ]
        
    def generate_samples(self, step, epoch):
        """Generate audio samples during training"""
        try:
            logger.info(f"Generating samples at step {step}, epoch {epoch}")
            
            for i, text in enumerate(self.sample_texts):
                # Generate audio
                output = synthesis(
                    model=self.model,
                    text=text,
                    CONFIG=self.config,
                    use_cuda=False,
                    use_griffin_lim=True,
                    speaker_id="benedict_cumberbatch",  # Should match your speaker
                    style_wav=self.style_reference_path,  # Add this
                )
                
                # Extract waveform from output
                wav = output['wav'] if isinstance(output, dict) else output
                # Save sample
                sample_path = self.samples_dir / f"sample_{epoch:03d}_{step:06d}_{i}.wav"
                self.ap.save_wav(wav, str(sample_path))
                
            logger.info(f"Samples saved to {self.samples_dir}")
            
        except Exception as e:
            logger.error(f"Error generating samples: {e}")

class CustomTrainer(Trainer):
    """Custom trainer with enhanced progress tracking"""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.progress_tracker = None
        self.epoch_progress = None
        
    def set_progress_tracker(self, tracker):
        self.progress_tracker = tracker
        
    def train_loop(self):
        """Override train loop to add progress tracking"""
        # Initialize progress bar for epoch
        self.epoch_progress = tqdm(total=len(self.train_loader), 
                                  desc=f"Epoch {self.epochs_done}/{self.config.epochs}",
                                  unit="batch")
        super().train_loop()
        self.epoch_progress.close()
        
    def train_step(self, batch, batch_n_steps, step, loader_start_time):
        """Fixed train step with progress tracking"""
        # Call parent train step
        outputs, loss = super().train_step(batch, batch_n_steps, step, loader_start_time)
        
        # Update progress bar
        if self.epoch_progress:
            self.epoch_progress.update(1)
            self.epoch_progress.set_postfix({"loss": f"{loss.item():.4f}"})
        
        # Generate samples at specified intervals
        if self.progress_tracker and step % self.config.sample_step == 0 and step > 0:
            self.progress_tracker.generate_samples(step, self.epochs_done)
            
        return outputs, loss
    

def load_pretrained_model():
    """Load pretrained model for fine-tuning"""
        # Download pretrained model
    manager = ModelManager()
    output_model_path, output_config_path, model_item = manager.download_model("tts_models/en/ljspeech/tacotron2-DDC")
    
    return output_config_path, output_model_path

def main():
    """Main training function with fine-tuning setup"""
    
    logger.info("Starting British Voice Fine-tuning Pipeline")
    logger.info("=" * 50)
    
    # Load config
    with open('/Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/config.json', 'r') as f:
        config_data = json.load(f)
    
    # Force CPU usage
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    torch.set_num_threads(16)  # Optimize for M3 Pro
    
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--restore_path', type=str, default=None, help='Path to specific checkpoint')
    args = parser.parse_args()
    
    # Setup paths
    output_path = Path(config_data['output_path'])
    output_path.mkdir(exist_ok=True)
    
    logger.info(f"Output directory: {output_path}")
    
    # Load pretrained model
    logger.info("Loading pretrained model...")
    pretrained_config_path, pretrained_model_path = load_pretrained_model()
    
    # Model config for fine-tuning
    config = Tacotron2Config()
    # load raw config and remove comments
    with open(pretrained_config_path, 'r') as f:
        config_str = f.read()
    config_str = re.sub(r'//.*', '', config_str)
    with open(pretrained_config_path, 'w') as f:
        f.write(config_str)
    config.load_json(str(pretrained_config_path))
    
    
    # Override with fine-tuning settings
    config.batch_size = config_data['batch_size']
    config.eval_batch_size = max(4, config_data['batch_size'] // 4)
    config.lr = config_data['learning_rate']
    config.warmup_steps = config_data['warmup_steps']
    config.epochs = config_data['epochs']
    config.print_step = config_data['print_step']
    config.save_step = config_data['save_step']
    config.eval_step = config_data['eval_step']
    config.sample_step = config_data['sample_step']
    config.weight_decay = config_data['weight_decay']
    config.grad_clip = config_data['grad_clip']
    config.mixed_precision = False
    
    # Audio settings
    config.audio.sample_rate = config_data['sample_rate']
    config.audio.do_trim_silence = True
    config.audio.trim_db = 60
    
    # Fine-tuning specific settings
    config.run_name = "benedict_voice_finetune"
    config.checkpoint = True
    config.tb_model_param_stats = True
    
    base_dataset_config = BaseDatasetConfig(
        formatter=custom_formatter,  # Use custom formatter
        language="en",
        path=config_data['audio_path'],
        meta_file_train=config_data['metadata_path'],
        meta_file_val=None,
    )
    
    # gst config
    config.use_gst = True
    config.gst = Tacotron2Config.gst.__class__()
    config.gst.gst_embedding_dim = 256
    config.gst.gst_num_heads = 4
    config.gst.gst_num_style_tokens = 10
    # Dataset config
    config.datasets = [base_dataset_config]
    
    # tensorboard config
    config.log_step = config_data['print_step']  # Log to tensorboard at print intervals
    config.tb_plot_step = config_data['sample_step']  # Plot alignments
    config.tb_model_param_stats = True  # Log model stats
    config.log_model_step = config.save_step  # Log model weights
    config.metrics = ["loss", "grad_norm", "alignment_score"]
    config.loss_masking = True

    logger.info("Loading dataset...")
    # Load samples
    train_samples, eval_samples = load_tts_samples(
        config.datasets,
        eval_split=True,
        eval_split_max_size=500,
        eval_split_size=config_data['eval_split_size'],
    )
    
    logger.info(f"Training samples: {len(train_samples)}")
    logger.info(f"Evaluation samples: {len(eval_samples)}")
    
    # Init audio processor
    ap = AudioProcessor.init_from_config(config)
    
    # Init tokenizer
    tokenizer, config = TTSTokenizer.init_from_config(config)
    
    # Init model
    model = Tacotron2(config, ap, tokenizer, speaker_manager=None)
    
    # Load pretrained weights for fine-tuning
    logger.info("Loading pretrained weights...")
    checkpoint = torch.load(pretrained_model_path, map_location='cpu')
    model.load_state_dict(checkpoint['model'], strict=False)
    logger.info("Pretrained weights loaded successfully")
    
    # Setup trainer args
    trainer_args = TrainerArgs()
    if args.restore_path:
        trainer_args.restore_path = args.restore_path
        logger.info(f"Using specified checkpoint: {args.restore_path}")
    else:
        # Auto-find latest checkpoint
        checkpoint_files = list(output_path.glob("checkpoint_*.pth"))
        if checkpoint_files:
            latest_checkpoint = max(checkpoint_files, key=os.path.getctime)
            trainer_args.restore_path = str(latest_checkpoint)
            logger.info(f"Auto-resuming from latest checkpoint: {latest_checkpoint}")
        else:
            trainer_args.restore_path = None
            logger.info("No checkpoints found, starting from scratch")
    trainer_args.skip_train_epoch = False
    trainer_args.start_with_eval = True
    trainer_args.grad_accum_steps = 1
    
    # Init custom trainer
    trainer = CustomTrainer(
        trainer_args,
        config,
        output_path,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
    )
    
    # Get first audio file for style reference
    style_reference = train_samples[0]["audio_file"]
    
    # Setup progress tracker WITH style reference
    progress_tracker = ProgressTracker(
        model, 
        config, 
        ap, 
        tokenizer, 
        output_path,
        style_reference_path=style_reference  # Add this
    )
    trainer.set_progress_tracker(progress_tracker)
    
    # Generate initial samples
    logger.info("Generating initial samples...")
    progress_tracker.generate_samples(0, 0)
    
    # Start fine-tuning
    logger.info("Starting fine-tuning...")
    logger.info(f"Training for {config.epochs} epochs")
    logger.info(f"Batch size: {config.batch_size}")
    logger.info(f"Learning rate: {config.lr}")
    
    start_time = time.time()
    
    try:
        trainer.fit()
    except KeyboardInterrupt:
        logger.info("Training interrupted by user")
    except Exception as e:
        logger.error(f"Training error: {e}")
        raise
    finally:
        # Final checkpoint and samples
        logger.info("Saving final checkpoint...")
        torch.save(model.state_dict(), output_path / "model.pth")
        
        logger.info("Generating final samples...")
        progress_tracker.generate_samples(trainer.total_steps_done, trainer.epochs_done)
        
        end_time = time.time()
        duration = end_time - start_time
        logger.info(f"Training completed in {duration/3600:.2f} hours")
        
        logger.info("Fine-tuning complete!")
        logger.info(f"Model saved to: {output_path}")
        logger.info(f"Samples saved to: {progress_tracker.samples_dir}")

if __name__ == "__main__":
    main()

'''
        
        script_path = Path(output_path) / 'finetune.py'
        with open(script_path, 'w') as f:
            f.write(training_script)
        
        print(f"Fine-tuning script saved to {script_path}")

def main():
    """Main execution function"""
    
    # Configuration - UPDATE THESE PATHS
    AUDIO_DIR = "/Users/ivkrasovskii/model-voice-generator/dataset/chunks"      # Directory with .wav files
    CSV_PATH = "/Users/ivkrasovskii/model-voice-generator/dataset/transcriptions.csv"  # CSV with file_name,transcription
    OUTPUT_DIR = "/Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune"
    
    print("Benedict Cumberbatch Voice Fine-tuning Pipeline")
    print("=" * 50)
    
    # Step 1: Process dataset
    print("Step 1: Processing dataset...")
    processor = AudioDataProcessor(AUDIO_DIR, CSV_PATH, OUTPUT_DIR)
    df = processor.process_dataset()
    
    if len(df) == 0:
        print("❌ No valid samples found! Check your audio directory and CSV file.")
        return
    
    # Dataset summary
    total_duration_hours = df['duration'].sum() / 3600
    print(f"\n📊 Dataset Summary:")
    print(f"  ✅ Valid samples: {len(df)}")
    print(f"  ⏱️  Total duration: {total_duration_hours:.2f} hours")
    print(f"  📈 Average duration: {df['duration'].mean():.2f} seconds")
    print(f"  📉 Min duration: {df['duration'].min():.2f} seconds")  
    print(f"  📊 Max duration: {df['duration'].max():.2f} seconds")
    
    # Check if we have enough data
    if total_duration_hours < 2:
        print("⚠️  Warning: Less than 2 hours of data. Consider adding more samples for better results.")
    
    # Step 2: Prepare for fine-tuning
    print("\nStep 2: Preparing for fine-tuning...")
    processor.prepare_for_coqui_tts(df)
    
    # Step 3: Setup fine-tuning
    print("\nStep 3: Setting up fine-tuning...")
    trainer = CoquiTTSFineTuner(f"{OUTPUT_DIR}/config.json")
    trainer.install_dependencies()
    trainer.create_training_script(OUTPUT_DIR)
    
    print("\n✅ Setup complete!")
    print("\n🚀 Next steps:")
    print("1. Install dependencies (run the pip commands shown above)")
    print("2. Update the paths in this script (AUDIO_DIR and CSV_PATH)")
    print("3. Start fine-tuning:")
    print(f"   cd {OUTPUT_DIR}")
    print("   python finetune.py")
    print("\n📝 Training Features:")
    print("  • Conservative fine-tuning to preserve base model quality")
    print("  • Progress tracking with minimal console output")  
    print("  • Regular checkpoints every 500 steps")
    print("  • Sample generation every ~half epoch")
    print("  • Lower learning rate to avoid breaking weights")
    print("  • CPU optimized for Mac M3 Pro")
    
    print(f"\n⚠️  Note: CPU training will take 12-48 hours for your dataset.")
    print("   Consider using Google Colab with GPU for 10x faster training.")
    
    # Show sample data
    print(f"\n📄 Sample processed data:")
    print(df.head(3)[['audio_path', 'text', 'duration']].to_string(index=False))

if __name__ == "__main__":
    main()

2025-06-26 22:34:01,022 - INFO - Loaded 1872 transcriptions from CSV
2025-06-26 22:34:01,023 - INFO - Processing audio files...


Benedict Cumberbatch Voice Fine-tuning Pipeline
Step 1: Processing dataset...


Processing files: 100%|██████████| 1872/1872 [00:06<00:00, 268.16it/s]
2025-06-26 22:34:08,005 - INFO - Valid samples: 1850 out of 1872
2025-06-26 22:34:08,021 - INFO - Manifest saved to /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/training_manifest.json
2025-06-26 22:34:08,047 - INFO - Coqui TTS metadata saved to /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/metadata.txt
2025-06-26 22:34:08,047 - INFO - Fine-tuning config saved to /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/config.json



📊 Dataset Summary:
  ✅ Valid samples: 1850
  ⏱️  Total duration: 15.41 hours
  📈 Average duration: 29.99 seconds
  📉 Min duration: 17.12 seconds
  📊 Max duration: 30.00 seconds

Step 2: Preparing for fine-tuning...

Step 3: Setting up fine-tuning...
Run these commands to install dependencies:
  pip install coqui-tts>=0.20.0
  pip install librosa soundfile pandas tqdm
  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
  pip install tensorboard
Fine-tuning script saved to /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/finetune.py

✅ Setup complete!

🚀 Next steps:
1. Install dependencies (run the pip commands shown above)
2. Update the paths in this script (AUDIO_DIR and CSV_PATH)
3. Start fine-tuning:
   cd /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune
   python finetune.py

📝 Training Features:
  • Conservative fine-tuning to preserve base model quality
  • Progress tracking with minimal con